# Import packages

In [5]:
import cv2
from keras.utils import load_img
from keras.saving import load_model
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

import segmenteverygrain as seg
import segmenteverygrain.interactions as si
import os
import rasterio

# %matplotlib qt

## Load models

Maybe you need to clone first SAM2. Then add the yaml file in the bottom of the next chunk

In [2]:
# UNET model
unet = load_model(
    "../models/seg_model.keras",
    custom_objects={"weighted_crossentropy": seg.weighted_crossentropy},
)

# Auto-detect device: CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback
import torch
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# SAM 2.1 checkpoints. Download from:
# https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
sam = build_sam2(r"C:\Users\leoni\sam2\sam2\configs\sam2.1\sam2.1_hiera_l.yaml", r"C:\Users\leoni\segmenteverygrain\models\sam2.1_hiera_large.pt", device=device)


## Set your directories

In [3]:
dir = os.chdir("C:/Users/leoni/kirgis_repo/segmenteverygrain/leonieexamples/test_if_mask_necessary")
dir = os.getcwd()
os.makedirs("ouputgpkg", exist_ok=True)
os.makedirs("inputtiles", exist_ok=True)
os.makedirs("plots", exist_ok=True)


inputtiledir = os.path.join(dir, "inputtiles/")
ouputgpkg = os.path.join(dir, "ouputgpkg/")

plotdir = os.path.join(dir, "plots/")

Input your masks of the flight

In [10]:
# ==========================================================
# MASKEN-SEKTION (ab hier)
# 1 = NICHT segmentieren
# 0 oder nodata = segmentieren erlaubt
# ==========================================================
veg_mask_path = r"C:\Users\leoni\kirgis_repo\veg-shad-wat-filtering\1_uav-filters\vegetation\veg_masks_rgbvi_fallback\25092025_Aktal_gravel_2_10m_OM_RGB_utm_coregistered_ws128_clipwater_vegmask_rgbvi.tif"
shadow_mask_path = r"C:\Users\leoni\kirgis_repo\veg-shad-wat-filtering\1_uav-filters\shadow\0-02\25092025_Aktal_gravel_2_10m_OM_RGB_utm_coregistered_ws128_clipwater_shadowmask.tif"

import glob
from rasterio.warp import reproject, Resampling

def read_mask_for_tile(tile_path, full_mask_src, mask_band=1):
    """
    Schneidet/projiziert die Vollmaske auf das Grid des Tiles.
    Ergebnis: 2D-Array mit exakt gleicher Höhe/Breite wie das Tile.
    """
    with rasterio.open(tile_path) as tsrc:
        tile_mask = np.zeros((tsrc.height, tsrc.width), dtype=np.float32)

        reproject(
            source=rasterio.band(full_mask_src, mask_band),
            destination=tile_mask,
            src_transform=full_mask_src.transform,
            src_crs=full_mask_src.crs,
            dst_transform=tsrc.transform,
            dst_crs=tsrc.crs,
            resampling=Resampling.nearest,   # wichtig für binäre Masken
            dst_nodata=np.nan
        )

    return tile_mask


# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))

print(f"Found {len(tiles)} tiles")

if len(tiles) == 0:
    raise RuntimeError("No tiles found.")

# --- read pixel size once from first tile ---
with rasterio.open(tiles[0]) as src:
    xres, yres = src.res          # CRS units per pixel (usually meters)
    crs = src.crs

gsd_units = float((xres + yres) / 2.0)
gsd_mm = gsd_units * 1000.0       # assumes CRS units are meters (e.g., UTM)

minor_mm = 20.0                   # 2 cm
minor_px = minor_mm / gsd_mm

# area threshold from minor-axis (circle assumption)
min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

print("CRS:", crs)
print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")


# --- loop (mit Masken + Shape-Fix + robustem labels-Handling) ---
with rasterio.open(veg_mask_path) as veg_src, rasterio.open(shadow_mask_path) as shadow_src:

    for i, fname in enumerate(tiles, start=1):
        tile_id = os.path.splitext(os.path.basename(fname))[0]
        print(f"[{i}/{len(tiles)}] {tile_id}")

        # load image
        image = np.array(load_img(fname))

        # passende Tile-Masken aus den Vollmasken holen
        veg_tile = read_mask_for_tile(fname, veg_src, mask_band=1)
        shadow_tile = read_mask_for_tile(fname, shadow_src, mask_band=1)

        # Deine Logik:
        # 1 = maskiert (nicht segmentieren)
        # 0 oder nodata = erlaubt
        veg_exclude = np.isclose(veg_tile, 1.0, equal_nan=False)
        shadow_exclude = np.isclose(shadow_tile, 1.0, equal_nan=False)

        exclude_mask = veg_exclude | shadow_exclude   # True = blockieren
        valid_mask = ~exclude_mask                    # True = segmentieren erlaubt

        # --------------------------------------------------
        # SHAPE CHECK / FIX (für TIFFs: H/W manchmal vertauscht)
        # --------------------------------------------------
        img_hw = image.shape[:2]      # (H, W)
        mask_hw = valid_mask.shape    # erwartet auch (H, W)

        if mask_hw != img_hw:
            if mask_hw[::-1] == img_hw:
                valid_mask = valid_mask.T
                print(f"Transposed valid_mask from {mask_hw} to {valid_mask.shape} to match image {img_hw}")
            else:
                raise ValueError(
                    f"Shape mismatch between image and valid_mask: "
                    f"image.shape[:2]={img_hw}, valid_mask.shape={mask_hw}"
                )

        if not np.any(valid_mask):
            print("Tile fully masked by vegetation/shadow -> skipping")
            continue

        # load + predict
        image_pred = seg.predict_image(image, unet, I=256)

        # Zusätzlicher Check: falls prediction-Shape abweicht
        pred_hw = image_pred.shape[:2]
        if pred_hw != valid_mask.shape:
            if valid_mask.shape[::-1] == pred_hw:
                valid_mask = valid_mask.T
                print(f"Transposed valid_mask to match image_pred: {pred_hw}")
            else:
                raise ValueError(
                    f"Shape mismatch between image_pred and valid_mask: "
                    f"image_pred.shape[:2]={pred_hw}, valid_mask.shape={valid_mask.shape}"
                )

        # Maskierte Flächen schon VOR Prompt-Erzeugung ausschließen
        image_pred = np.array(image_pred, copy=True)
        image_pred[~valid_mask] = 0

        # Optional/empfohlen: Bild dort auch neutralisieren
        image_for_seg = np.array(image, copy=True)
        image_for_seg[~valid_mask] = 0

        # prompts (Unet -> points)
        labels, coords = seg.label_grains(image_for_seg, image_pred, dbs_max_dist=10.0)

        # --------------------------------------------------
        # Sicherheitsfilter: Prompts in maskierten Bereichen entfernen
        # WICHTIG: labels kann 2D Label-Bild sein -> nicht immer mit keep filtern!
        # --------------------------------------------------
        coords = np.asarray(coords)

        if len(coords) > 0:
            # Annahme: coords = (x, y)
            x_idx = np.clip(np.round(coords[:, 0]).astype(int), 0, valid_mask.shape[1] - 1)
            y_idx = np.clip(np.round(coords[:, 1]).astype(int), 0, valid_mask.shape[0] - 1)

            keep = valid_mask[y_idx, x_idx]
            coords = coords[keep]

            # labels nur filtern, wenn es wirklich ein 1D Prompt-Labelarray ist
            try:
                labels_arr = np.asarray(labels)
                if labels_arr.ndim == 1 and len(labels_arr) == len(keep):
                    labels = labels_arr[keep]
                # sonst labels unverändert lassen (z.B. 2D Labelbild)
            except Exception:
                pass

        if len(coords) == 0:
            print("No valid prompts after masking -> skipping tile")
            continue

        # SAM segmentation
        all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
            sam,
            image_for_seg,
            image_pred,
            coords,
            labels,
            min_area=min_area_px,
            plot_image=True,
            remove_edge_grains=True,
            remove_large_objects=True,
        )

        # Optional: finale Maske hart beschneiden (zusätzliche Sicherheit)
        try:
            mask_all = np.array(mask_all, copy=True)
            if mask_all.shape != valid_mask.shape:
                if mask_all.shape[::-1] == valid_mask.shape:
                    mask_all = mask_all.T
                else:
                    print(f"Warning: mask_all shape {mask_all.shape} != valid_mask shape {valid_mask.shape}")
            mask_all[~valid_mask] = 0
        except Exception as e:
            print(f"Could not apply final hard mask cut: {e}")

        # save plot
        figname = os.path.join(plotdir, f"{tile_id}.png")
        fig.savefig(figname, dpi=200, bbox_inches="tight")
        plt.close(fig)

        print("Saved the Plot")
        print("done with segmentation")

Found 7 tiles
CRS: EPSG:32643
Pixel size: 0.003501 units/px (~3.501 mm/px)
2 cm => minor_px=5.71 px -> min_area_px=25.6 px^2
[1/7] tile_r000006_c000024
segmenting image tiles...


100%|██████████| 3/3 [00:01<00:00,  2.41it/s]


creating masks using SAM...


100%|██████████| 645/645 [00:45<00:00, 14.23it/s]


finding overlapping polygons...


...

finding overlapping polygons...


1555it [27:35,  1.06s/it]


finding best polygons...


  0%|          | 0/3 [13:40<?, ?it/s]


KeyboardInterrupt: 

## Not including Masks

## Run segmentation

Collect tiles

In [14]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img
import rasterio

# --- collect tiles ---
tiles = sorted(glob.glob(os.path.join(inputtiledir, "*.tif")))

print(f"Found {len(tiles)} tiles")


if len(tiles) == 0:
    raise RuntimeError("No tiles found.")

# --- read pixel size once from first tile ---
with rasterio.open(tiles[0]) as src:
    xres, yres = src.res          # CRS units per pixel (usually meters)
    crs = src.crs

gsd_units = float((xres + yres) / 2.0)
gsd_mm = gsd_units * 1000.0       # assumes CRS units are meters (e.g., UTM)

minor_mm = 20.0                   # 2 cm
minor_px = minor_mm / gsd_mm

# area threshold from minor-axis (circle assumption)
min_area_px = float(np.pi * (minor_px / 2.0) ** 2)

print("CRS:", crs)
print(f"Pixel size: {gsd_units:.6f} units/px (~{gsd_mm:.3f} mm/px)")
print(f"2 cm => minor_px={minor_px:.2f} px -> min_area_px={min_area_px:.1f} px^2")



# --- loop ---
for i, fname in enumerate(tiles, start=1):
    tile_id = os.path.splitext(os.path.basename(fname))[0]
    print(f"[{i}/{len(tiles)}] {tile_id}")

    # load + predict
    image = np.array(load_img(fname))
    image_pred = seg.predict_image(image, unet, I=256)

    # prompts (Unet -> points)
    labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=10.0)

    # SAM segmentation
    all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
        sam,
        image,
        image_pred,
        coords,
        labels,
        min_area=min_area_px,
        plot_image=True,
        remove_edge_grains=True,
        remove_large_objects=True,
    )
    figname =os.path.join(plotdir, f"{tile_id}.png")

    fig.savefig(figname, dpi = 200, bbox_inches="tight")
    plt.close(fig)
    print("Saved the Plot")
    print("done with segmentation")






Found 1 tiles
CRS: EPSG:32643
Pixel size: 0.003501 units/px (~3.501 mm/px)
2 cm => minor_px=5.71 px -> min_area_px=25.6 px^2
[1/1] tile_r000008_c000026
segmenting image tiles...


100%|██████████| 3/3 [00:02<00:00,  1.24it/s]


creating masks using SAM...


100%|██████████| 822/822 [01:33<00:00,  8.80it/s]


finding overlapping polygons...


...

finding best polygons...


100%|██████████| 200/200 [00:32<00:00,  6.17it/s]


creating labeled image...


100%|██████████| 244/244 [00:01<00:00, 128.43it/s]


Saved the Plot
done with segmentation


In [10]:
min_area_px

25.62514189487139